# ML Teaching Essentials — Model Diagnostics
## TL;DR — Plain English Introduction

**What separates good ML engineers from great ones?** Knowing WHY a model fails and HOW to fix it. This notebook teaches the diagnostic skills that interviewers test.

**The 5 most common ML failures (and fixes):**
1. **High bias (underfitting)** — model too simple → use more complex model or add features
2. **High variance (overfitting)** — model memorizes training data → add regularization, more data, dropout
3. **Vanishing gradients** — gradients near 0 in deep networks → use ReLU, batch norm, residual connections
4. **Exploding gradients** — gradients blow up → gradient clipping (`clip_grad_norm_`)
5. **Data leakage** — test data contaminated training → strict train/val/test splits before ANY processing

**No ML background?**
- Training loss = how wrong is my model on data it has seen?
- Validation loss = how wrong on data it has NOT seen?
- If train loss << val loss: overfitting (memorizing, not learning)
- If both losses are high: underfitting (not learning at all)

**Learning path:** 00/02 (ML fundamentals) → 00/06 (PyTorch) → This notebook → 05/01 (Deep learning) → 10/01 (Fine-tuning)

## Beginner Teaching Frame

**Who should fully work through this notebook:** anyone who can train a model but does not yet trust their diagnostic judgment.

**How to study it on a first pass:**
- pause after each plot and say what failure pattern it reveals
- connect every visualization to a training decision
- treat this notebook as a lens for reading later notebooks more intelligently

**Common traps:** optimizing metrics blindly, ignoring validation behavior, and confusing a good-looking training curve with a good model.


## Canonical Resource Upgrade

Strong companion references:
- [scikit-learn model evaluation docs](https://scikit-learn.org/stable/model_selection.html)
- [Dive into Deep Learning](https://d2l.ai/)
- [Made With ML](https://madewithml.com/) for practical debugging habits


# ML Teaching Essentials — Model Diagnostics & Training

## What This Notebook Covers

These are the concepts that separate someone who *uses* ML from someone who *understands* it. Every professional ML interview will probe at least one of these. They were missing from the other notebooks — this one covers them all, with visualizations and biological examples.

## Learning Objectives
- [ ] Draw and interpret learning curves — diagnose overfitting vs underfitting
- [ ] Demonstrate the bias-variance tradeoff with actual data
- [ ] Compare L1 vs L2 vs no regularization on a real dataset
- [ ] Build early stopping from scratch and with callbacks
- [ ] Implement learning rate scheduling (step decay, cosine annealing)
- [ ] Plot multi-class confusion matrix and ROC curves
- [ ] Compute SHAP values and interpret individual predictions
- [ ] Debug 5 common training failures (NaN loss, stuck training, oscillating loss, etc.)
- [ ] Run a hyperparameter sensitivity analysis with plots

## Prerequisites
- ML Fundamentals (Notebook 00/02)
- NumPy + Matplotlib basics

## 🗺️ Prerequisites & Learning Path

| | |
|--|--|
| 🔑 Prerequisites | 00/02 (NumPy/matplotlib), 05/01 (deep learning training fundamentals) |
| 📍 You Are Here | Module 09/01 — Model Diagnostics & Training |
| ➡️ Next Steps | Any module requiring model training — this is a standalone reference |
| 🏁 Goal | Diagnose underfitting/overfitting from learning curves; compare regularization strategies; implement gradient flow monitoring |

### 🆕 From Scratch? Start Here:
1. [StatQuest bias-variance video](https://www.youtube.com/watch?v=EuBBz3bI-aA) — 10 min, best visual explanation
2. [sklearn learning_curve docs](https://scikit-learn.org/stable/modules/learning_curve.html) — hands-on starting point
3. 00/02 (this repo) — NumPy/matplotlib for plotting diagnostics
4. This notebook — model diagnostics
**Time:** 8–12h | **Difficulty:** ⭐⭐⭐

### 🔗 Cross-References
- Builds on: 00/02 (plotting), 05/01 (training loop patterns), 04/01 (ML fundamentals — bias-variance introduced there)
- Used by: ALL modules with trained models — apply these diagnostics to every notebook's training cell
- Standalone: This notebook can be read independently as a reference guide

In [ ]:
# Prerequisites cell — imports and setup
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression, make_classification
from sklearn.model_selection import train_test_split, learning_curve, validation_curve
from sklearn.linear_model import Ridge, Lasso, LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
import torch
import torch.nn as nn
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("All imports successful!")
print("Module 09: Model Diagnostics — bias-variance, regularization, gradient flow, SHAP")

## 1. Bias-Variance Tradeoff — Visual Demonstration

**Bias** = how wrong is your model on average? (underfitting)
**Variance** = how much does your model change with different training data? (overfitting)

The tradeoff:
- Simple model → high bias, low variance (underfits — misses the pattern)
- Complex model → low bias, high variance (overfits — memorizes noise)
- **Goal**: find the sweet spot

```
Error
  │
  │\  total error
  │ \ ← you want this valley
  │  \_____/
  │   bias²   variance
  └──────────────────── Model Complexity
```

In [ ]:
# Bias-Variance Tradeoff — visual demonstration
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

np.random.seed(42)
X_true = np.linspace(0, 1, 100)
y_true = np.sin(2 * np.pi * X_true)
X_train = np.random.uniform(0, 1, 30)
y_train = np.sin(2 * np.pi * X_train) + np.random.normal(0, 0.3, 30)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
degrees = [1, 5, 20]
for ax, d in zip(axes, degrees):
    model = Pipeline([('poly', PolynomialFeatures(d)), ('lr', LinearRegression())])
    model.fit(X_train.reshape(-1,1), y_train)
    y_pred = model.predict(X_true.reshape(-1,1))
    ax.scatter(X_train, y_train, alpha=0.5, s=20, label='Training data')
    ax.plot(X_true, y_true, 'g-', label='True function')
    ax.plot(X_true, y_pred, 'r-', label=f'Degree {d}')
    cv = cross_val_score(model, X_train.reshape(-1,1), y_train, cv=5, scoring='neg_mean_squared_error')
    ax.set_title(f'Degree {d}: CV MSE={-cv.mean():.3f}')
    ax.legend(fontsize=7)
    ax.set_ylim(-2, 2)

plt.tight_layout()
plt.savefig('/tmp/bias_variance.png', dpi=72)
print("Degree  1: HIGH bias, low variance (underfitting)")
print("Degree  5: balanced bias-variance")
print("Degree 20: low bias, HIGH variance (overfitting)")

## 2. Learning Curves — Diagnose Your Model

A **learning curve** plots train and validation performance vs. **training set size**.

| Pattern | Diagnosis | Fix |
|---------|----------|-----|
| Both curves low, converging | Underfitting (high bias) | More complex model, more features |
| Train high, val low, big gap | Overfitting (high variance) | More data, regularization, dropout |
| Train high, val high, converging | Good fit | Add more data to push val higher |
| Both curves low, not converging | Need more data | More data always helps here |

In [ ]:
# Learning curves — diagnose your model
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import learning_curve
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

np.random.seed(42)
X, y = make_classification(n_samples=1000, n_features=20, n_informative=5, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (model, name) in zip(axes, [
    (LogisticRegression(max_iter=1000), 'Logistic Regression (low capacity)'),
    (SVC(C=10, kernel='rbf'), 'SVM RBF (high capacity)')
]):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y, cv=5, n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, 10),
        scoring='accuracy'
    )
    ax.plot(train_sizes, train_scores.mean(axis=1), 'b-o', label='Train', markersize=4)
    ax.plot(train_sizes, val_scores.mean(axis=1),   'r-o', label='Validation', markersize=4)
    ax.fill_between(train_sizes, train_scores.mean(axis=1)-train_scores.std(axis=1),
                                  train_scores.mean(axis=1)+train_scores.std(axis=1), alpha=0.1, color='b')
    ax.fill_between(train_sizes, val_scores.mean(axis=1)-val_scores.std(axis=1),
                                  val_scores.mean(axis=1)+val_scores.std(axis=1), alpha=0.1, color='r')
    ax.set_title(name)
    ax.set_xlabel('Training set size')
    ax.set_ylabel('Accuracy')
    ax.legend(); ax.set_ylim(0.5, 1.0)
plt.tight_layout()
plt.savefig('/tmp/learning_curves.png', dpi=72)
print("High train/val gap → overfitting (needs more data or regularization)")
print("Low train AND val → underfitting (needs more capacity)")

## 3. Regularization — L1 vs L2 vs No Regularization

| Type | Penalty | Effect | Best for |
|------|---------|--------|----------|
| **None** | 0 | Overfits on small datasets | Very large datasets |
| **L2 (Ridge)** | λ·Σw² | Shrinks all weights toward 0 | Correlated features (genes!) |
| **L1 (Lasso)** | λ·Σ\|w\| | **Drives some weights to exactly 0** (feature selection) | Sparse signal (few real genes) |
| **ElasticNet** | α·L1 + (1-α)·L2 | Best of both | Grouped correlated features |

In [ ]:
# Regularization — L1 vs L2 vs no regularization
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_regression

np.random.seed(42)
X, y, true_coef = make_regression(n_samples=100, n_features=50, n_informative=10,
                                   coef=True, random_state=42)
scaler = StandardScaler()
X_s = scaler.fit_transform(X)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
alphas = np.logspace(-3, 3, 100)

for ax, (ModelClass, name) in zip(axes, [
    (Ridge, 'Ridge (L2)'),
    (Lasso, 'Lasso (L1)'),
    (None, 'Coef comparison')
]):
    if ModelClass is not None:
        coefs = []
        for a in alphas:
            m = ModelClass(alpha=a)
            m.fit(X_s, y)
            coefs.append(m.coef_)
        coefs = np.array(coefs)
        for i in range(min(15, coefs.shape[1])):
            ax.plot(np.log10(alphas), coefs[:, i], alpha=0.5)
        ax.set_xlabel('log10(alpha)')
        ax.set_ylabel('Coefficient value')
        ax.set_title(f'{name} regularization path')
        ax.axvline(0, color='k', linestyle='--', alpha=0.3)
    else:
        ridge = Ridge(alpha=1.0).fit(X_s, y)
        lasso = Lasso(alpha=0.1).fit(X_s, y)
        ax.scatter(range(50), true_coef/true_coef.max(), s=20, label='True', alpha=0.7)
        ax.scatter(range(50), ridge.coef_/abs(ridge.coef_).max(), s=20, label='Ridge', alpha=0.7)
        ax.scatter(range(50), lasso.coef_/max(abs(lasso.coef_).max(),1e-10), s=20, label='Lasso', alpha=0.7)
        ax.legend(); ax.set_title('Coef comparison')
        non_zero_lasso = (np.abs(lasso.coef_) > 1e-6).sum()
        print(f"Lasso non-zero features: {non_zero_lasso}/50")
plt.tight_layout()
plt.savefig('/tmp/regularization.png', dpi=72)

## 4. Confusion Matrix Heatmap & Multi-class ROC Curves

In [ ]:
# Confusion matrix + multi-class ROC
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from sklearn.preprocessing import label_binarize
import seaborn as sns

np.random.seed(42)
X, y = make_classification(n_samples=500, n_features=20, n_classes=3,
                            n_informative=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)

print(classification_report(y_test, y_pred))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

y_bin = label_binarize(y_test, classes=[0,1,2])
colors = ['blue','red','green']
for i, c in enumerate(colors):
    fpr, tpr, _ = roc_curve(y_bin[:,i], y_proba[:,i])
    axes[1].plot(fpr, tpr, color=c, label=f'Class {i} (AUC={auc(fpr,tpr):.2f})')
axes[1].plot([0,1],[0,1],'k--'); axes[1].set_title('Multi-class ROC')
axes[1].legend(); axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
plt.tight_layout()
plt.savefig('/tmp/confusion_roc.png', dpi=72)

## 5. Feature Importance: Gini vs Permutation vs SHAP

| Method | What it measures | Pitfall |
|--------|-----------------|--------|
| **Gini importance** | Average impurity decrease when splitting on feature | Biased toward high-cardinality features |
| **Permutation importance** | Drop in score when feature is randomly shuffled | Slower but unbiased; works with any model |
| **SHAP** | Contribution of each feature to each individual prediction | Most interpretable; uses game theory |

In [ ]:
# Feature importance: Gini vs Permutation vs SHAP
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

np.random.seed(42)
X, y = make_classification(n_samples=500, n_features=10, n_informative=5, random_state=42)
feature_names = [f'feat_{i}' for i in range(10)]

clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(X, y)

# Gini importance
gini_imp = clf.feature_importances_

# Permutation importance
perm = permutation_importance(clf, X, y, n_repeats=10, random_state=42)
perm_imp = perm.importances_mean

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
idx_gini = np.argsort(gini_imp)[::-1]
idx_perm = np.argsort(perm_imp)[::-1]

axes[0].bar(range(10), gini_imp[idx_gini])
axes[0].set_xticks(range(10))
axes[0].set_xticklabels([feature_names[i] for i in idx_gini], rotation=45, ha='right')
axes[0].set_title('Gini (MDI) Importance')

axes[1].bar(range(10), perm_imp[idx_perm])
axes[1].set_xticks(range(10))
axes[1].set_xticklabels([feature_names[i] for i in idx_perm], rotation=45, ha='right')
axes[1].set_title('Permutation Importance')

plt.tight_layout()
plt.savefig('/tmp/feature_importance.png', dpi=72)
print("Gini: fast, biased toward high-cardinality features")
print("Permutation: model-agnostic, captures interactions")
print("SHAP: local + global, satisfies Shapley axioms (install: pip install shap)")

## 6. Early Stopping & Learning Rate Scheduling (PyTorch)

**Early stopping**: Monitor validation loss. If it doesn't improve for `patience` epochs → stop.
**Why**: Prevents overfitting without knowing in advance how many epochs to train.

**Learning rate scheduling**: Change LR during training.
- Start high → converge fast
- Reduce → fine-tune near minimum
- Common schedules: ReduceOnPlateau, CosineAnnealing, StepLR

In [ ]:
# Early stopping + LR scheduling
import torch
import torch.nn as nn
import numpy as np
import copy

class EarlyStopping:
    def __init__(self, patience=10, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best = float('inf')
        self.counter = 0
        self.best_state = None

    def __call__(self, val_loss, model):
        if val_loss < self.best - self.min_delta:
            self.best = val_loss
            self.counter = 0
            self.best_state = copy.deepcopy(model.state_dict())
            return False
        self.counter += 1
        return self.counter >= self.patience

# Training with warmup + cosine decay
torch.manual_seed(42)
model = nn.Sequential(nn.Linear(20,64), nn.ReLU(), nn.Linear(64,1))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def get_lr(step, warmup=100, total=500):
    if step < warmup:
        return step / warmup
    progress = (step - warmup) / (total - warmup)
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda s: get_lr(s))
es = EarlyStopping(patience=5)

X = torch.randn(200, 20); y = torch.randn(200, 1)
lrs = []
for step in range(50):
    optimizer.zero_grad()
    loss = nn.MSELoss()(model(X[:16]), y[:16])
    loss.backward(); optimizer.step(); scheduler.step()
    lrs.append(optimizer.param_groups[0]['lr'])
    if es(loss.item(), model):
        print(f"Early stop at step {step}"); break
print(f"LR at step 0: {lrs[0]:.6f}, step 10: {lrs[10]:.6f}, step 49: {lrs[-1]:.6f}")
print("Warmup prevents large updates on random initialization")

## 7. Gradient Flow Analysis

**Vanishing gradients**: gradients near 0 in early layers → they don't learn.
**Exploding gradients**: gradients very large → unstable training, NaN loss.

Causes:
- Deep networks without residual connections
- Wrong weight initialization (too large or too small)
- Saturating activations (sigmoid, tanh) vs non-saturating (ReLU)

In [ ]:
# Gradient flow analysis
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

def plot_grad_flow(named_params):
    ave_grads, max_grads, layers = [], [], []
    for n, p in named_params:
        if p.requires_grad and p.grad is not None and 'bias' not in n:
            layers.append(n.split('.')[0])
            ave_grads.append(p.grad.abs().mean().item())
            max_grads.append(p.grad.abs().max().item())
    return layers, ave_grads, max_grads

torch.manual_seed(42)
# Deep network without residual connections
model = nn.Sequential(*[nn.Linear(64, 64) if i%2==0 else nn.Tanh()
                         for i in range(20)] + [nn.Linear(64, 1)])
x = torch.randn(16, 64)
y = torch.randn(16, 1)
loss = nn.MSELoss()(model(x), y)
loss.backward()

layers, avg_grads, max_grads = plot_grad_flow(model.named_parameters())
print("Gradient norms per layer (first 5 and last 5):")
for l, a, m in zip(layers[:5], avg_grads[:5], max_grads[:5]):
    print(f"  {l}: avg={a:.2e} max={m:.2e}")
print("  ...")
for l, a, m in zip(layers[-3:], avg_grads[-3:], max_grads[-3:]):
    print(f"  {l}: avg={a:.2e} max={m:.2e}")

ratio = avg_grads[0] / (avg_grads[-1] + 1e-12)
print(f"\nFirst/last gradient ratio: {ratio:.2e}")
print("Ratio >> 1 → vanishing gradients in early layers")

## 8. The Debugging Guide — 5 Common Training Failures

Every ML practitioner must recognize these patterns.

In [ ]:
# Debugging guide — 5 common training failures
import torch
import torch.nn as nn
import numpy as np

print("DEBUGGING GUIDE: 5 Common Training Failures")
print("=" * 60)

failures = {
    "1. Loss doesn't decrease": {
        "Symptoms": "Loss stays constant or oscillates",
        "Causes": ["LR too high/low", "Wrong loss function", "Data not normalized", "Bug in forward pass"],
        "Debug": "Print loss every step; check grad norms; visualize predictions"
    },
    "2. NaN/Inf loss": {
        "Symptoms": "loss=nan after a few steps",
        "Causes": ["LR too large", "Log(0) in loss", "Gradient explosion"],
        "Debug": "Clip gradients; add eps to log; reduce LR; check for -inf logits"
    },
    "3. Overfitting": {
        "Symptoms": "Train loss ↓, Val loss ↑",
        "Causes": ["Too many parameters", "Too few training samples", "No regularization"],
        "Debug": "Add dropout/L2; more data/augmentation; early stopping; smaller model"
    },
    "4. Underfitting": {
        "Symptoms": "Both train and val loss high",
        "Causes": ["Model too small", "LR too low", "Too few epochs", "Feature engineering needed"],
        "Debug": "Increase model capacity; tune LR; train longer; add features"
    },
    "5. Class imbalance": {
        "Symptoms": "High accuracy but poor recall for minority class",
        "Causes": ["Imbalanced labels (e.g., 95% negative)"],
        "Debug": "Use class_weight, SMOTE, focal loss; check per-class metrics"
    },
}

for name, info in failures.items():
    print(f"\n{name}")
    print(f"  Symptoms: {info['Symptoms']}")
    print(f"  Causes: {'; '.join(info['Causes'])}")
    print(f"  Debug: {info['Debug']}")

# Quick diagnostic function
def diagnose_training(losses):
    if any(np.isnan(l) for l in losses):
        return "NaN loss detected — check LR, log(0), gradient explosion"
    if losses[-1] > losses[0] * 0.99:
        return "Loss not decreasing — check LR, normalization"
    if len(losses) > 5 and losses[-1] > min(losses) * 1.1:
        return "Loss increasing — possible overfitting or unstable training"
    return "Training looks healthy"

print("\nDiagnostic examples:")
for desc, losses in [
    ("Healthy", [1.0, 0.8, 0.6, 0.5, 0.45, 0.42]),
    ("Not improving", [1.0, 1.01, 0.99, 1.0, 1.02]),
    ("NaN", [1.0, 0.8, float('nan')])
]:
    try:
        print(f"  {desc}: {diagnose_training(losses)}")
    except:
        print(f"  {desc}: NaN detected")

## 9. Hyperparameter Sensitivity Analysis

**Validation curve**: vary ONE hyperparameter, plot train vs val score.
Use `sklearn.model_selection.validation_curve` for clean implementation.

In [ ]:
# Hyperparameter sensitivity analysis
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.svm import SVC
from sklearn.model_selection import validation_curve, cross_val_score

np.random.seed(42)
X, y = make_classification(n_samples=500, n_features=20, n_informative=10, random_state=42)

# Validation curve: vary C in SVM
C_range = np.logspace(-3, 3, 20)
train_scores, val_scores = validation_curve(
    SVC(kernel='rbf'), X, y, param_name='C', param_range=C_range, cv=5, scoring='accuracy'
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(C_range, train_scores.mean(axis=1), 'b-o', label='Train', markersize=4)
ax.semilogx(C_range, val_scores.mean(axis=1),   'r-o', label='Validation', markersize=4)
ax.fill_between(C_range, train_scores.mean(1)-train_scores.std(1),
                           train_scores.mean(1)+train_scores.std(1), alpha=0.1, color='b')
ax.fill_between(C_range, val_scores.mean(1)-val_scores.std(1),
                           val_scores.mean(1)+val_scores.std(1), alpha=0.1, color='r')
ax.set_xlabel('C (regularization)'); ax.set_ylabel('Accuracy')
ax.set_title('Validation Curve: SVM Hyperparameter C')
ax.legend()
plt.tight_layout()
plt.savefig('/tmp/validation_curve.png', dpi=72)

best_C = C_range[val_scores.mean(axis=1).argmax()]
print(f"Best C: {best_C:.4f}")
print(f"Best CV accuracy: {val_scores.mean(axis=1).max():.3f}")
print("Low C → high bias (underfitting); High C → high variance (overfitting)")

## 📚 Resources

### 1️⃣ Theory — Foundations & Math
- **Bias-variance decomposition**: E[MSE] = Bias² + Variance + σ²(irreducible noise) — formal proof via bias-variance expansion
- **Learning curve theory**: PAC learning sample complexity — generalization error ≤ ε with probability ≥ 1-δ requires n ≥ (1/ε)(ln|H| + ln(1/δ)) samples
- **Regularization as MAP estimation**: L2 penalty ↔ Gaussian prior on weights; L1 penalty ↔ Laplace prior — Bayesian interpretation
- **Information criteria**: AIC = 2k - 2ln(L̂), BIC = k·ln(n) - 2ln(L̂) — penalize complexity differently for model selection
- **Dropout**: stochastic depth regularization, equivalent to ensemble of 2^n sub-networks; NO dropout at test time

### 2️⃣ Must-Have Popular Resources
- 📘 Elements of Statistical Learning ch. 7 (Hastie et al., free PDF) — model assessment, cross-validation, AIC/BIC
- 📘 Understanding Deep Learning (Simon Prince, free 2024) — modern DL theory with bias-variance for neural nets
- 🎓 StatQuest bias-variance videos (Josh Starmer) — the clearest visual intuition available
- ⭐ GitHub [scikit-learn learning_curve](https://scikit-learn.org/stable/modules/learning_curve.html) — built-in diagnostic utilities
- ⭐ GitHub [wandb/wandb](https://github.com/wandb/wandb) 10k★ — Weights & Biases experiment tracking

### 3️⃣ Practicals — Hands-On
- 💻 Generate learning curves for 5 models (linear, tree, RF, SVM, MLP) — diagnose each as high-bias or high-variance
- 💻 Compare L1 / L2 / Dropout regularization on the same dataset — plot coefficient paths and validation loss
- 💻 Implement gradient flow monitoring — log mean |∇W| per layer, detect vanishing/exploding gradients
- 🤗 HuggingFace: [wandb/wandb](https://docs.wandb.ai/) — integrate with PyTorch training loop in 3 lines
- 📊 Kaggle: [House Prices](https://www.kaggle.com/c/house-prices-advanced-regression-techniques) — complete model diagnostics pipeline

### 4️⃣ Real-World Problems
- 🏭 Genentech / Roche / Novartis: all ML teams use learning curve analysis for go/no-go decisions before scaling compute
- 🏭 Pharma ML: bias-variance framework determines whether to collect more data or regularize harder for biomarker models
- 📊 Dataset: [Kaggle House Prices](https://www.kaggle.com/c/house-prices-advanced-regression-techniques) — 79 features, 1460 samples; classic for diagnostics
- 🔬 Application: Clinical outcome prediction — learning curves reveal if 500 patients is enough to train a reliable model

### 5️⃣ Interview Tips
- ❓ Must know: Diagnose from learning curve — high training error + high val error = underfitting (add complexity); low training error + high val error = overfitting (regularize/more data)
- ❓ Must know: Why NO dropout at test time — training uses E[stochastic masks]; test uses full network scaled by (1-p) to match expected activation
- ❓ Must know: High training loss + high val loss means → underfitting (bias-dominated), not overfitting
- ⚠️ Common mistake: Concluding "need more data" from a learning curve that has already plateaued — check if val curve has converged
- 💡 Pro tip: Always plot BOTH training and validation curves — reporting only val curve hides whether the model is underfitting

### 6️⃣ Hot Industry Topics
- 🔥 Trending: [wandb/wandb](https://github.com/wandb/wandb) — experiment tracking, now standard in industry ML teams
- 🔥 Trending: [optuna/optuna](https://github.com/optuna/optuna) 10k★ — Bayesian hyperparameter optimization (replaces grid search)
- 🔥 Trending: [Lightning-AI/pytorch-lightning](https://github.com/Lightning-AI/pytorch-lightning) — clean training loops with built-in diagnostics
- 🚀 Build this: A model diagnostic dashboard that generates learning curves, regularization paths, and gradient flow plots for any sklearn/PyTorch model in one function call

### Time Complexity — Diagnostic Operations
| Diagnostic | Time | Space | Notes |
|-----------|------|-------|-------|
| Learning curve (k points) | O(k × training) | O(model) | sklearn's `learning_curve()` |
| Validation curve (k values) | O(k × training) | O(model) | `validation_curve()` |
| Cross-validation (k folds) | O(k × training) | O(model) | `cross_val_score()` |
| Gradient flow analysis | O(parameters) | O(layers) | hook on `.grad` |
| SHAP values (tree) | O(n × features) | O(n) | TreeSHAP algorithm |
| SHAP values (deep) | O(n × background) | O(background) | DeepSHAP |
| Confusion matrix | O(n) | O(classes²) | — |
| ROC curve | O(n log n) | O(n) | sort by score |

In [ ]:
# Resources for 09/01
print("KEY RESOURCES — Module 09/01: Model Diagnostics")
print()
refs = [
    "Bias-variance decomposition: Bishop PRML Chapter 3",
    "Learning curve tutorial: scikit-learn.org/stable/modules/learning_curve",
    "Understanding SHAP: christophm.github.io/interpretable-ml-book/shap",
    "Gradient flow: Andrej Karpathy — 'A Recipe for Training Neural Networks' (blog)",
    "Debugging ML: fast.ai Lesson 7",
    "Hyperparameter tuning: Optuna, Ray Tune, W&B Sweeps",
    "StatQuest: youtube.com/@statquest — bias-variance, ROC curves, regularization",
]
for r in refs:
    print(f"  {r}")

## 10. Interview Questions & Answers

**Q1: What is the difference between bias and variance? How do you reduce each?**

Bias = error from wrong assumptions in the model (e.g., linear model for non-linear data). Reduces with: more complex model, more features, less regularization.
Variance = error from sensitivity to training data fluctuations. Reduces with: more data, regularization (L1/L2/dropout), ensemble methods.

**Q2: Your model has 98% train accuracy but 60% val accuracy. What's wrong and how do you fix it?**

Severe overfitting. Fixes in priority order: (1) add regularization (L2, dropout), (2) get more data, (3) reduce model complexity (fewer layers/trees), (4) feature selection to remove noise features, (5) early stopping.

**Q3: When should you use L1 vs L2 regularization for gene expression data?**

L1 (Lasso) when you believe only a few genes are truly informative (sparse signal) — it drives uninformative gene weights to exactly 0. L2 (Ridge) when many genes contribute small signals (dense signal, e.g., polygenic traits). ElasticNet combines both — good default for genomics.

**Q4: Explain learning rate scheduling. Why is cosine annealing popular?**

LR scheduling changes the learning rate during training. Start high (explore) → decay (exploit local minimum). Cosine annealing smoothly decreases from LR_max to 0 following a cosine curve — avoids sudden LR drops that can destabilize training near convergence. Used in: ResNet, BERT, ESM-2, AlphaFold.

**Q5: What is gradient clipping and when do you need it?**

Cap the gradient norm: `clip_grad_norm_(params, max_norm=1.0)`. Needed for: RNNs (sequence length amplifies gradients), transformer models on long sequences, and whenever you see NaN loss or oscillating training. Not needed for simple feed-forward networks with proper initialization.

**Q6: Your loss is NaN after epoch 3. Walk me through your debugging process.**

1. Check input data: `assert not np.any(np.isnan(X))` — NaN in input propagates everywhere
2. Check labels: off-by-one errors cause log(0) in cross-entropy
3. Check learning rate: reduce by 10x
4. Add gradient clipping: `clip_grad_norm_(model.parameters(), 1.0)`
5. Add `torch.autograd.detect_anomaly()` context to find the exact operation that produces NaN
6. Check for inf in model outputs: `torch.isfinite(logits).all()`

## Real World Problems

### Problem 1: Model Diagnostics on TCGA Cancer Data
Train a gene expression classifier on the TCGA PANCAN dataset. Plot learning curves, bias-variance curves (vary max_depth of RF), and feature importance. Which genes are most important? Do they match known oncogenes?

**Resources**: [TCGA PANCAN (Kaggle)](https://www.kaggle.com/datasets/crawford/gene-expression-cancer-rna-seq) | [scikit-learn diagnostics (GitHub)](https://github.com/scikit-learn/scikit-learn)

### Problem 2: Train a Protein Classifier with Early Stopping
Fine-tune ESM-2 (or build a CNN) on a protein localization dataset. Implement early stopping with patience=10. Compare: no scheduling vs cosine annealing vs ReduceOnPlateau. Which converges fastest and reaches highest val accuracy?

**Resources**: [DeepLoc protein localization (HuggingFace)](https://huggingface.co/datasets/tattabio/protein_localization) | [ESM-2 model (HuggingFace)](https://huggingface.co/facebook/esm2_t6_8M_UR50D)

### Problem 3: Kaggle — Interpret ML Predictions
Use the CAFA5 protein function prediction competition. Train a baseline model, then use permutation importance and SHAP to identify which protein features (sequence, structure, evolutionary) are most predictive. Compare SHAP values across different GO term categories.

**Resources**: [CAFA5 (Kaggle)](https://www.kaggle.com/competitions/cafa-5-protein-function-prediction) | [SHAP library (GitHub)](https://github.com/shap/shap)

## Mastery Check

Before moving on, make sure you can:
1. distinguish underfitting, overfitting, and data leakage
2. explain what a learning curve tells you
3. diagnose one likely intervention from a plot
4. use this notebook to critique a later modeling result


---
## 🔍 SHAP — Model Interpretability for Scientific ML

SHAP (SHapley Additive exPlanations) is the gold standard for explaining ML models in drug discovery, clinical genomics, and structural biology. It answers: **"Why did this model predict this value for this sample?"**


In [ ]:
# SHAP INTERPRETABILITY — Complete Implementation
# Even without the shap package, we implement TreeSHAP approximation manually
# to understand the concept, then show the real shap API

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

# Load a real dataset: breast cancer (real medical features)
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Train a Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_tr, y_tr)
print(f"Random Forest accuracy: {rf.score(X_te, y_te):.3f}")

# ── Manual permutation-based feature importance (SHAP concept) ──
def permutation_importance(model, X, y, n_repeats=10):
    """
    Permutation importance: for each feature, shuffle it and measure accuracy drop.
    This is a simplified version of SHAP marginal contribution calculation.
    """
    baseline = model.score(X, y)
    importances = {}
    for col in X.columns:
        drops = []
        for _ in range(n_repeats):
            X_perm = X.copy()
            X_perm[col] = X_perm[col].sample(frac=1, random_state=42).values
            drops.append(baseline - model.score(X_perm, y))
        importances[col] = np.mean(drops)
    return importances

perm_imp = permutation_importance(rf, X_te, y_te, n_repeats=5)
sorted_imp = sorted(perm_imp.items(), key=lambda x: -x[1])

print("\nTop 10 features by permutation importance:")
for feat, imp in sorted_imp[:10]:
    bar = '█' * int(imp * 200)
    print(f"  {feat:<35}: {imp:+.4f}  {bar}")

# Try real SHAP if available
try:
    import shap
    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_te)

    # For binary classification, shap_values[1] = values for class 1 (benign)
    sv = shap_values[1] if isinstance(shap_values, list) else shap_values

    # Beeswarm plot (most informative SHAP plot)
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # 1. Mean absolute SHAP value (global importance)
    mean_abs_shap = np.abs(sv).mean(axis=0)
    top10_idx = np.argsort(mean_abs_shap)[-10:][::-1]
    top10_names = [data.feature_names[i] for i in top10_idx]
    top10_vals = mean_abs_shap[top10_idx]

    axes[0].barh(range(10), top10_vals[::-1], color='steelblue')
    axes[0].set_yticks(range(10))
    axes[0].set_yticklabels(top10_names[::-1])
    axes[0].set_xlabel('Mean |SHAP value|')
    axes[0].set_title('Global Feature Importance (SHAP)')

    # 2. SHAP values for first test sample (local explanation)
    sample_idx = 0
    sample_shap = sv[sample_idx]
    top5_local_idx = np.argsort(np.abs(sample_shap))[-5:][::-1]
    colors = ['#d62728' if sample_shap[i] > 0 else '#1f77b4' for i in top5_local_idx]

    axes[1].barh(range(5), sample_shap[top5_local_idx][::-1], color=colors[::-1])
    axes[1].set_yticks(range(5))
    axes[1].set_yticklabels([data.feature_names[i] for i in top5_local_idx][::-1])
    axes[1].axvline(0, color='black', linewidth=0.8)
    axes[1].set_xlabel('SHAP value (contribution to prediction)')
    axes[1].set_title(f'Local Explanation — Sample {sample_idx}\n'
                      f'True: {"Benign" if y_te.iloc[0] else "Malignant"}, '
                      f'Pred: {"Benign" if rf.predict(X_te.iloc[[0]])[0] else "Malignant"}')

    plt.tight_layout()
    plt.savefig('shap_plots.png', dpi=100)
    plt.show()
    print("\n[Real SHAP plots generated]")

except ImportError:
    print("\nshap not installed. Install with: pip install shap")
    print("Manual permutation importance plotted above instead.")

    fig, ax = plt.subplots(figsize=(9, 6))
    features = [f for f, _ in sorted_imp[:10]][::-1]
    vals = [v for _, v in sorted_imp[:10]][::-1]
    colors = ['#d62728' if v > 0 else '#1f77b4' for v in vals]
    ax.barh(features, vals, color=colors)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Mean accuracy drop when feature is shuffled')
    ax.set_title('Permutation Feature Importance\n(proxy for SHAP global importance)')
    plt.tight_layout()
    plt.savefig('permutation_importance.png', dpi=100)
    plt.show()

---
## 🎯 Optuna — Hyperparameter Optimization

Manual grid search is inefficient. Optuna uses Bayesian optimization (TPE) to find good hyperparameters in far fewer trials than grid search.


In [ ]:
# OPTUNA HYPERPARAMETER OPTIMIZATION
# Optuna: https://optuna.org — the standard tool for ML hyperparameter search

import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import GradientBoostingClassifier

np.random.seed(42)
X, y = make_classification(n_samples=1000, n_features=20, n_informative=10, random_state=42)

# ── Manual baseline: try 5 configs by hand ──
manual_configs = [
    {'n_estimators': 50,  'max_depth': 3, 'learning_rate': 0.1},
    {'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.05},
    {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.01},
    {'n_estimators': 100, 'max_depth': 7, 'learning_rate': 0.1},
    {'n_estimators': 150, 'max_depth': 5, 'learning_rate': 0.05},
]

print("MANUAL SEARCH (5 configs):")
best_manual = 0
for cfg in manual_configs:
    clf = GradientBoostingClassifier(**cfg, random_state=42)
    score = cross_val_score(clf, X, y, cv=5).mean()
    print(f"  {cfg} → {score:.4f}")
    best_manual = max(best_manual, score)
print(f"Best manual: {best_manual:.4f}")

# ── Optuna search ──
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 300),
            'max_depth': trial.suggest_int('max_depth', 2, 8),
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        }
        clf = GradientBoostingClassifier(**params, random_state=42)
        return cross_val_score(clf, X, y, cv=5).mean()

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=30, show_progress_bar=False)

    print(f"\nOPTUNA SEARCH (30 trials):")
    print(f"  Best score: {study.best_value:.4f}")
    print(f"  Best params: {study.best_params}")
    print(f"  Improvement over manual: +{(study.best_value - best_manual)*100:.2f}%")

    # Plot optimization history
    import matplotlib.pyplot as plt
    values = [t.value for t in study.trials]
    best_so_far = np.maximum.accumulate(values)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(values, 'o', alpha=0.4, color='steelblue', markersize=4, label='Trial score')
    ax.plot(best_so_far, '-', color='red', linewidth=2, label='Best so far')
    ax.axhline(best_manual, color='gray', linestyle='--', label=f'Best manual ({best_manual:.4f})')
    ax.set_xlabel('Trial number')
    ax.set_ylabel('CV Score (5-fold)')
    ax.set_title('Optuna Optimization History\n(TPE searches intelligently, not randomly)')
    ax.legend()
    plt.tight_layout()
    plt.savefig('optuna_history.png', dpi=100)
    plt.show()

except ImportError:
    print("\nOptuna not installed. Install with: pip install optuna")
    print()
    print("OPTUNA KEY CONCEPTS:")
    print("  - Trial: one evaluation of the objective function")
    print("  - Study: collection of trials with direction (maximize/minimize)")
    print("  - TPE sampler: Tree-structured Parzen Estimator — models P(good|params)")
    print("  - After ~20 trials, Optuna concentrates search on promising regions")
    print()
    print("WHEN TO USE OPTUNA vs GRID SEARCH:")
    print("  Grid search: 5+ hours for 3 params × 5 values each (5^3=125 configs)")
    print("  Optuna: 30 trials, usually finds 90% of the grid search best in <1 hour")
    print("  Rule: if any param is continuous or > 10 values → always use Optuna")

In [ ]:
# PROBABILITY CALIBRATION — Critical for Scientific Applications
# A well-calibrated model: if it says 80% probability, 80% of those predictions are correct
# Uncalibrated model: 80% probability predictions might be correct only 60% of the time

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.model_selection import train_test_split

np.random.seed(42)
X, y = make_classification(n_samples=3000, n_features=20, n_informative=10,
                            n_clusters_per_class=2, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

# Train several models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),  # Well-calibrated by design
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),  # Overconfident
    'SVM (sigmoid)': SVC(probability=True, kernel='rbf'),      # Poorly calibrated
    'RF + Platt': CalibratedClassifierCV(RandomForestClassifier(n_estimators=100, random_state=42),
                                         method='sigmoid', cv=5),
}

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flat

for ax, (name, model) in zip(axes, models.items()):
    model.fit(X_tr, y_tr)
    probs = model.predict_proba(X_te)[:, 1]

    # Calibration curve (reliability diagram)
    fraction_pos, mean_pred = calibration_curve(y_te, probs, n_bins=10)

    ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Perfect calibration')
    ax.plot(mean_pred, fraction_pos, 's-', linewidth=2, markersize=6,
            label=f'{name}', color='steelblue')

    # Expected Calibration Error (ECE) — lower is better
    ece = np.mean(np.abs(fraction_pos - mean_pred))
    ax.set_title(f'{name}\nECE = {ece:.4f}', fontsize=10)
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Fraction of positives')
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Probability Calibration (Reliability Diagrams)\n'
             'Points on the diagonal = perfectly calibrated', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('calibration_curves.png', dpi=100)
plt.show()

print("CALIBRATION RESULTS:")
print("  Logistic Regression: well-calibrated (ECE ≈ 0.01-0.03)")
print("  Random Forest: overconfident — probabilities pushed toward 0 and 1")
print("  SVM: poor calibration (Platt scaling was designed to fix this)")
print("  RF + Platt: improved calibration via post-hoc scaling")
print()
print("WHY CALIBRATION MATTERS IN SCIENTIFIC APPLICATIONS:")
print("  Drug discovery: p=0.8 binding prediction → reserve for experiment 8 vs 5 candidates")
print("  Clinical: p=0.75 disease risk → risk stratification decisions")
print("  Structure prediction: confidence scores guide which predictions to validate")
print()
print("WHEN TO CALIBRATE:")
print("  Always calibrate tree models (RF, GBM) for probability outputs")
print("  Logistic Regression is calibrated by default (it minimizes log-loss)")
print("  Neural networks: use temperature scaling (T-scaling) post-hoc")

---
## 🔎 Diagnose This Training Run

Here are four training curves with known issues. Identify the problem for each before looking at the diagnosis.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simulate 4 classic failure patterns
epochs = range(1, 51)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
patterns = []

# Pattern 1: Overfitting
np.random.seed(1)
tr1 = 0.95 - 0.3 * np.exp(-np.arange(50)/5) + np.random.normal(0, 0.005, 50)
va1 = 0.70 + 0.1 * np.exp(-np.arange(50)/8) - 0.02 * np.arange(50)/50 + np.random.normal(0, 0.01, 50)
patterns.append((tr1, va1, "TRAINING RUN A", "?"))

# Pattern 2: Underfitting (model too small / too much regularization)
np.random.seed(2)
tr2 = 0.65 - 0.05 * np.exp(-np.arange(50)/20) + np.random.normal(0, 0.005, 50)
va2 = 0.62 - 0.03 * np.exp(-np.arange(50)/20) + np.random.normal(0, 0.007, 50)
patterns.append((tr2, va2, "TRAINING RUN B", "?"))

# Pattern 3: Learning rate too high (unstable training)
np.random.seed(3)
base = 0.75 * (1 - np.exp(-np.arange(50)/3))
tr3 = base + 0.15 * np.sin(np.arange(50)/3) + np.random.normal(0, 0.03, 50)
va3 = base * 0.95 + 0.10 * np.sin(np.arange(50)/3 + 0.5) + np.random.normal(0, 0.03, 50)
tr3 = np.clip(tr3, 0.3, 0.98)
va3 = np.clip(va3, 0.3, 0.95)
patterns.append((tr3, va3, "TRAINING RUN C", "?"))

# Pattern 4: Good training (reference)
np.random.seed(4)
tr4 = 0.92 - 0.45 * np.exp(-np.arange(50)/8) + np.random.normal(0, 0.005, 50)
va4 = 0.87 - 0.35 * np.exp(-np.arange(50)/6) + np.random.normal(0, 0.007, 50)
patterns.append((tr4, va4, "TRAINING RUN D", "?"))

for ax, (tr, va, title, _) in zip(axes.flat, patterns):
    ax.plot(epochs, tr, 'b-', linewidth=2, label='Train accuracy')
    ax.plot(epochs, va, 'r-', linewidth=2, label='Val accuracy')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.set_title(title, fontweight='bold')
    ax.legend()
    ax.set_ylim(0.2, 1.0)
    ax.grid(True, alpha=0.3)

plt.suptitle('DIAGNOSE THESE TRAINING RUNS\nBefore scrolling: what is wrong with each?',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('training_diagnosis.png', dpi=100)
plt.show()

print("\nSTOP. Before reading the diagnosis:")
print("  1. Look at each plot and write down your diagnosis.")
print("  2. For each: what is the problem, and what would you do to fix it?")
print()
print("."*50)
print()
print("DIAGNOSES:")
print()
print("RUN A — OVERFITTING:")
print("  Signal: Train acc >> Val acc, val acc decreasing after epoch 20")
print("  Evidence: Widening gap between train and val curves")
print("  Fixes: Add dropout, reduce model capacity, more training data, early stopping")
print()
print("RUN B — UNDERFITTING:")
print("  Signal: Both train and val accuracy plateau at low values (~65%)")
print("  Evidence: No gap between train/val (model hasn't even memorized training set)")
print("  Fixes: Increase model capacity, reduce regularization, train longer, better features")
print()
print("RUN C — LEARNING RATE TOO HIGH:")
print("  Signal: Oscillating curves — improvement then crash, repeatedly")
print("  Evidence: High variance between adjacent epochs; no smooth convergence")
print("  Fixes: Reduce learning rate by 10×, add gradient clipping, use LR scheduler")
print()
print("RUN D — HEALTHY TRAINING (reference):")
print("  Signal: Both curves converge smoothly, train slightly above val, stable plateau")
print("  Evidence: Small train-val gap, no oscillation, clear convergence")
print("  Action: Only add early stopping to prevent wasting compute after epoch 35")

## 🐛 Debug Exercise — Model Diagnostics

Find and fix the **3 bugs** in the model diagnostics code below.

**Expected correct output:**
```
Learning curve: train and validation scores plotted correctly with CV
Validation curve: testing 'C' parameter for LogisticRegression
Confusion matrix: rows = actual class, columns = predicted class
```

<details>
<summary>Show answer</summary>

**Bug 1 — Learning curve with wrong scoring:** `scoring=None` defaults to the estimator's
`score` method (accuracy). For imbalanced data this is misleading. Fix: `scoring='f1_macro'`.

**Bug 2 — Validation curve with wrong param_name:** `param_name='max_iter'` tests convergence
iterations, not model complexity. For LogisticRegression the complexity parameter is `'C'`.
Fix: `param_name='C'`, `param_range=[0.001, 0.01, 0.1, 1, 10, 100]`.

**Bug 3 — Confusion matrix axis labels swapped:** `sklearn`'s `confusion_matrix(y_true, y_pred)`
returns a matrix where **rows** are true labels and **columns** are predicted labels.
The code labels them the other way around.
Fix: xlabel = 'Predicted label', ylabel = 'True label'.
</details>


In [ ]:
# DEBUG EXERCISE — Model diagnostics, find and fix 3 bugs
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import learning_curve, validation_curve
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

X, y = make_classification(n_samples=300, n_features=10, weights=[0.7, 0.3], random_state=42)

# BUG 1: scoring=None uses accuracy — misleading for imbalanced classes
# Fix: scoring='f1_macro'
train_sizes, train_scores, val_scores = learning_curve(
    LogisticRegression(max_iter=1000),
    X, y,
    scoring=None,        # BUG: should be 'f1_macro'
    cv=5,
    train_sizes=np.linspace(0.1, 1.0, 5)
)
print(f"Learning curve val scores (accuracy, possibly misleading): {val_scores.mean(axis=1).round(3)}")

# BUG 2: validation curve testing max_iter instead of model complexity parameter C
# Fix: param_name='C', param_range=[0.001, 0.01, 0.1, 1, 10, 100]
param_range = [10, 50, 100, 200, 500]
train_vc, val_vc = validation_curve(
    LogisticRegression(),
    X, y,
    param_name='max_iter',    # BUG: tests convergence, not complexity
    param_range=param_range,
    scoring='f1_macro',
    cv=5
)
print(f"\nValidation curve (testing max_iter — not model complexity!)")
print(f"Val scores: {val_vc.mean(axis=1).round(3)}")
print("Expected: U-shaped curve showing bias-variance tradeoff (need C, not max_iter)")

# BUG 3: confusion matrix axis labels swapped
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
clf = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
y_pred = clf.predict(X_te)
cm = confusion_matrix(y_te, y_pred)

fig, ax = plt.subplots()
disp = ConfusionMatrixDisplay(cm)
disp.plot(ax=ax)
# BUG: labels are swapped — sklearn CM rows=actual, cols=predicted
ax.set_xlabel('True label')       # WRONG — should be 'Predicted label'
ax.set_ylabel('Predicted label')  # WRONG — should be 'True label'
plt.savefig('/tmp/cm_debug.png')
print("\nConfusion matrix saved to /tmp/cm_debug.png")
print("BUG: x-axis labeled 'True label', y-axis labeled 'Predicted label' — these are swapped!")


## ✏️ Exercise: Diagnose a Broken Model

A collaborator trained a model to predict patient outcomes from hospital EHR data. The results look odd:

| Split | Accuracy |
|-------|----------|
| Training | 98% |
| Validation (same hospital) | 62% |
| Test (new hospital) | 71% |

**Anomaly:** the test accuracy (71%) is *higher* than validation (62%). This is unusual — normally train ≥ val ≥ test.

**Your task:** write a diagnostic checklist. For each of the 5 experiments below:
1. State what you would measure / plot.
2. State what you would *expect to see* if this is the cause of the problem.
3. State what fix you would apply.

Then implement the `check_leakage` and `learning_curve_diagnosis` functions in the code cell below.

**The 5 experiments:**
1. Check for data leakage (target-encoded features, future data).
2. Plot a learning curve (train/val accuracy vs. training set size).
3. Check for distribution shift between hospitals (feature histograms or MMD).
4. Inspect the confusion matrix for class imbalance artefacts.
5. Verify that the validation split is truly held out (no patient appears in both train and val).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.datasets import make_classification

np.random.seed(42)

# ── Simulate the problematic scenario ────────────────────────────────
# Hospital A data: 1000 samples, 20 features (5 informative)
X_A, y_A = make_classification(
    n_samples=1000, n_features=20, n_informative=5,
    n_redundant=3, random_state=0)

# Inject data leakage: add the label (with small noise) as a feature
leakage_feature = y_A + np.random.randn(len(y_A)) * 0.01
X_A_leaky = np.column_stack([X_A, leakage_feature])  # 21 features

# Hospital B data: different distribution (covariate shift)
# Simulate covariate shift by using a different random_state and
# adding a global offset to all features.
X_B, y_B = make_classification(
    n_samples=300, n_features=20, n_informative=5,
    n_redundant=3, random_state=99)
X_B += 2.0  # shift all features by 2 to simulate distribution shift
X_B_leaky = np.column_stack([X_B, np.random.randn(len(X_B)) * 0.5])  # no leak

X_train, X_val, y_train, y_val = train_test_split(
    X_A_leaky, y_A, test_size=0.2, random_state=42)

model = GradientBoostingClassifier(n_estimators=100, max_depth=4,
                                   random_state=0)
model.fit(X_train, y_train)

train_acc = model.score(X_train, y_train)
val_acc   = model.score(X_val, y_val)
test_acc  = model.score(X_B_leaky, y_B)

print(f'Train accuracy:      {train_acc:.1%}')
print(f'Validation accuracy: {val_acc:.1%}')
print(f'Test accuracy:       {test_acc:.1%}')
print()


# ── TODO: Implement the diagnostic functions ──────────────────────────

def check_leakage(model, X_val, y_val, leakage_col_idx=-1):
    """
    Experiment 1: check for data leakage.
    Strategy: retrain WITHOUT the suspected leakage column and compare accuracy.
    Returns: (acc_with_leak, acc_without_leak)
    """
    # TODO: fit a new GradientBoostingClassifier on X_train with
    # leakage_col_idx column removed, score on X_val with same column removed.
    acc_with    = model.score(X_val, y_val)
    acc_without = None  # TODO
    return acc_with, acc_without


def learning_curve_diagnosis(X_train, y_train, X_val, y_val):
    """
    Experiment 2: plot learning curve.
    Shows whether the gap between train and val closes with more data.
    """
    # TODO: use sklearn's learning_curve() to get train/val scores
    # at sizes [0.1, 0.2, 0.4, 0.6, 0.8, 1.0] of the training set.
    # Plot mean ± std.
    pass


# ── Run diagnostics ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Experiment 1: leakage check
acc_w, acc_wo = check_leakage(model, X_val, y_val)
axes[0].bar(['With leak', 'Without leak'],
            [acc_w, acc_wo if acc_wo else 0],
            color=['firebrick', 'steelblue'], alpha=0.8)
axes[0].set_ylim(0, 1)
axes[0].set_title('Experiment 1: Leakage Check\n'
                  '(large drop = leakage confirmed)')
axes[0].set_ylabel('Validation accuracy')

# Experiment 3: feature distribution shift (Hospital A vs B)
for feat_idx in range(3):  # first 3 features
    axes[1].hist(X_A[:, feat_idx], bins=30, alpha=0.4,
                 label=f'Hospital A feat {feat_idx}')
    axes[1].hist(X_B[:, feat_idx], bins=30, alpha=0.4,
                 label=f'Hospital B feat {feat_idx}')
axes[1].set_title('Experiment 3: Distribution Shift\n'
                  '(overlap = no shift; separation = shift)')
axes[1].legend(fontsize=7)

# Experiment 4: confusion matrix
y_pred = model.predict(X_val)
cm = confusion_matrix(y_val, y_pred)
ConfusionMatrixDisplay(cm).plot(ax=axes[2], colorbar=False)
axes[2].set_title('Experiment 4: Confusion Matrix')

plt.suptitle('Model Diagnostic Checklist — 5 Experiments', fontweight='bold')
plt.tight_layout()
plt.show()

print('DIAGNOSTIC CHECKLIST:')
print('  Exp 1 — Leakage:       large accuracy drop when leakage feature removed?')
print('  Exp 2 — Learning curve: does val accuracy plateau far below train?')
print('  Exp 3 — Shift:         do Hospital A and B features overlap?')
print('  Exp 4 — Class imbal:   confusion matrix skewed to one class?')
print('  Exp 5 — Data split:    do patient IDs appear in both train and val?')

# ── SOLUTION HINT ─────────────────────────────────────────────────────
# check_leakage:
#   cols = [i for i in range(X_train.shape[1]) if i != leakage_col_idx]
#   m2 = GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=0)
#   m2.fit(X_train[:, cols], y_train)
#   acc_without = m2.score(X_val[:, cols], y_val)
#   return model.score(X_val, y_val), acc_without
#
# learning_curve_diagnosis:
#   from sklearn.model_selection import learning_curve as lc
#   train_sizes, train_scores, val_scores = lc(
#       GradientBoostingClassifier(random_state=0), X_train, y_train,
#       train_sizes=np.linspace(0.1, 1.0, 6), cv=3, scoring='accuracy')
#   plt.plot(train_sizes, train_scores.mean(1), label='train')
#   plt.plot(train_sizes, val_scores.mean(1), label='val')
#   plt.legend(); plt.xlabel('Training size'); plt.ylabel('Accuracy')
